# 데이터 수집부터 저장까지 — 파이프라인 완성

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🧩 데이터 수집부터 저장까지 — 파이프라인 완성

### — 모듈 종합 프로젝트: 8일간의 랭글링 기법을 한 흐름으로 잇다 —

---

## 📋 학습 목표

이 자습서를 마치면 여러분은 다음을 할 수 있습니다.

1. CSV의 한계를 이해하고, Parquet의 컬럼 저장·압축·스키마 보존 특성이 왜 분석 현장에서 표준이 되었는지 설명할 수 있습니다.
2. 결측·이상·중복·타입 불일치를 한 번에 진단하는 **데이터 품질 리포트 함수**를 직접 작성할 수 있습니다.
3. 비즈니스 데이터셋의 품질 문제를 진단하고, **왜 이 처리 방법을 골랐는지** 판단 근거를 함께 정리할 수 있습니다.
4. 인제스트(읽기) → 품질 진단 → 정제 → 변환 → 저장으로 이어지는 **하나의 파이프라인 함수**를 `.pipe()` 또는 단일 함수로 묶어낼 수 있습니다.
5. 같은 데이터를 CSV와 Parquet으로 저장해 보고, 용량·속도·타입 보존 측면에서 차이를 직접 비교할 수 있습니다.
6. 완성한 파이프라인 노트북을 **개인 GitHub 포트폴리오**에 정리해 동료가 읽을 수 있는 산출물로 제출할 수 있습니다.
7. "왜 이 결정을 내렸는가"를 코드 옆 글로 남기는 습관을 들일 수 있습니다.

> 💡 위 목록을 천천히 읽고, 지금 내가 자신 있는 것과 아직 낯선 것을 마음속으로 표시해보세요. 8일간의 데이터 랭글링이 오늘 어떤 모양으로 합쳐지는지 끝에서 다시 비교해보면 좋습니다.

## 📚 목차

| Part | 내용 | 핵심 질문 |
| --- | --- | --- |
| Part 0 | 종합 프로젝트 상황과 8일간의 흐름 정리 | 오늘 우리는 무엇을 한 흐름으로 묶을까요? |
| Part 1 | 인제스트와 CSV의 한계 | 원본 데이터를 어떻게 받아오고, 왜 CSV로는 부족할까요? |
| Part 2 | 데이터 품질 리포트 함수 | 결측·이상·중복·타입을 어떻게 자동으로 진단할까요? |
| Part 3 | 정제 단계 — 결측·이상치·문자열·날짜 통합 | 8일간 배운 정제 기법을 어떻게 한 줄기로 잇나요? |
| Part 4 | 변환 단계 — 병합·집계·파생 | 정제된 데이터를 어떻게 분석 가능한 형태로 바꿀까요? |
| Part 5 | 저장 단계 — Parquet vs CSV | 같은 데이터, 왜 Parquet으로 저장할까요? |
| Part 6 | 파이프라인 함수화 — `.pipe()`로 한 흐름 | 전체 과정을 어떻게 재사용 가능한 한 덩어리로 만들까요? |
| Part 7 | 판단 근거 문서화 | 코드 옆에 어떤 글을 함께 남겨야 동료가 읽을 수 있을까요? |
| 종합 실습 (종합 프로젝트) | 새 오염 데이터셋 처음부터 끝까지 | 새 데이터를 받아도 똑같은 흐름을 적용할 수 있을까요? |
| 정리 | 핵심 요약 · 다음 시간 | 오늘 완성한 파이프라인이 다음 모듈로 어떻게 이어질까요? |

## 종합 프로젝트 상황과 학습 지도

# 0. 종합 프로젝트 상황과 학습 지도

## 지난 8일간 무엇을 했나요?

지난 시간들에서 우리는 데이터를 **하나씩** 손봤습니다.

- **D+002** — `shape`·`info`·`describe`로 데이터를 *진찰*했고,
- **D+003** — 결측치와 이상치를 *판단해서 처리*했으며,
- **D+004** — 흩어진 테이블을 *병합·집계* 했고,
- **D+005** — 문자열과 날짜를 *쓸 수 있는 형태*로 정제했습니다.
- **D+006** — 같은 작업을 함수·`.pipe()`로 *자동화* 했고,
- **D+007** — 대용량에서도 버티는 도구(Polars)를 *선택*하는 법을 배웠습니다.
- **D+008** — 처리 결과를 *눈으로 검증·전달*하는 시각화를 익혔습니다.

여덟 번의 학습이 모두 **랭글링이라는 한 그림의 다른 조각**이었습니다. 오늘은 그 조각들을 **한 장의 파이프라인**으로 잇습니다.

## 오늘의 여정

오늘은 분석가가 새 데이터를 받았을 때 실제로 수행하는 **종단(end-to-end)** 흐름을 직접 만듭니다.

```text
[Part 1] 인제스트          원본 파일 읽기 · CSV의 한계
   ↓
[Part 2] 품질 리포트       결측·이상·중복·타입을 한 함수로 진단
   ↓
[Part 3] 정제              결측·이상치·문자열·날짜를 한 줄기로 처리
   ↓
[Part 4] 변환              병합·집계·파생으로 분석 가능 형태로 바꾸기
   ↓
[Part 5] 저장              Parquet로 결과 저장 + CSV와 비교
   ↓
[Part 6] 파이프라인 함수   .pipe()로 전 과정을 한 함수에 묶기
   ↓
[Part 7] 문서화            판단 근거를 코드 옆에 글로 남기기
   ↓
[종합 프로젝트] 새 오염 데이터셋을 처음부터 끝까지 혼자 처리
```

## 이 자습서 사용법

이 노트북은 **혼자서도 끝까지 학습할 수 있도록** 만들었습니다. 다음 네 박자로 진행하세요.

- 📖 **읽고** — 개념 설명과 판단 근거를 천천히 읽습니다.
- 💻 **실행하고** — 코드 셀을 위에서부터 순서대로 실행합니다. (단축키: `Shift + Enter`)
- ✏️ **고쳐보고** — "스스로 해보자!" 칸에서 코드를 직접 바꿔봅니다.
- 🤔 **답해보고** — 체크포인트 질문에 스스로 답해봅니다. 틀려도 괜찮습니다.

> ⚠ **주의:** 종합 프로젝트은 **앞 셀의 결과를 이어 받으며** 진행됩니다. 위에서부터 순서대로 실행해 주세요. 중간을 건너뛰면 "앞에서 만든 변수가 없다"는 오류가 납니다. 오류는 잘못이 아니라 *어느 단계가 끊겼는지* 알려주는 단서입니다.

> 💡 오늘은 **새 기법**보다 **결정의 흐름**을 배웁니다. 코드 자체는 8일간 본 것의 재조합입니다. 무엇을 선택하고, 왜 선택했는지에 집중하세요.

## 오늘의 무대: 모두마켓 종합 프로젝트 데이터셋

여러분은 모두마켓의 데이터 분석가입니다. 분기 종료를 앞두고 동료가 이렇게 말합니다.

> "지난 분기 주문·고객·로그 데이터를 다음 분기 KPI 분석가에게 넘겨야 하는데, 정제부터 저장까지 한 번에 돌아가는 파이프라인이 필요해요. 다음에 같은 일을 또 해야 할 테니까요."

이 한마디에 오늘 학습이 다 들어 있습니다.

- "한 번에 돌아가는" → **파이프라인 함수**가 필요합니다.
- "다음에 또" → **재사용 가능**해야 하고, **판단 근거가 기록**되어 있어야 합니다.
- "넘긴다" → 저장 포맷이 **타입을 보존**해야 합니다(Parquet).

이번에도 일부러 **지저분한** 데이터를 받았다고 가정합니다. 결측·이상치·문자열 오염·날짜 포맷 혼재·중복이 골고루 섞여 있습니다. 하나하나는 8일간 봤던 것들입니다. 오늘은 그것들을 **한 줄기로** 처리합니다.

> **🎯 오늘의 핵심**
> **결정을 코드 옆에 글로 남깁니다.** 동료가 다음 분기에 이 파이프라인을 읽을 때 "왜 이 값을 버렸지?"가 보여야 합니다.

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3
저장 폴더: C:\Users\USER\ai-data-bootcamp\D009\d009_outputs


In [2]:
# ─────────────────────────────────────────────
# 종합 프로젝트용 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '오염'을 일부러 심어 둡니다: 결측·이상치·표기 혼재·날짜 포맷 혼재·중복·타입 불일치)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 500
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(36, 10, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
        np.random.randint(0, 700, n_customers), unit="D"
    ),
})
# 오염 심기: 나이 이상치, 결측, 지역/멤버십 표기 혼재
customers.loc[[5, 6], "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[[20, 21, 22, 23], "gender"] = np.nan
customers.loc[30, "region"] = " 서울 "
customers.loc[31, "region"] = "Seoul"
customers.loc[40, "membership"] = "VIP"
customers.loc[41, "membership"] = " premium"

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 60
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 199900], n_products),
})

# 3) 주문(orders)
n_orders = 5000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05]),
})

# 날짜 포맷 혼재(문자열로 저장 — 일부러 통일하지 않음)
base_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_orders), unit="D")
date_strings = []
for i, d in enumerate(base_dates):
    if i % 3 == 0:
        date_strings.append(d.strftime("%Y-%m-%d"))
    elif i % 3 == 1:
        date_strings.append(d.strftime("%Y/%m/%d"))
    else:
        date_strings.append(d.strftime("%Y%m%d"))
orders["order_date"] = date_strings

# 오염 심기: 금액 결측, 수량 이상치, 중복 행, 음수 금액(환불로 보이는 값)
orders.loc[np.random.choice(n_orders, 80, replace=False), "amount"] = np.nan
orders.loc[7, "quantity"] = 100
orders.loc[8, "amount"] = -50000.0   # 환불 또는 입력 실수 (판단 필요)
orders = pd.concat([orders, orders.iloc[[0, 1, 2]]], ignore_index=True)  # 중복 3건

# 4) 웹 로그(web_logs) — 텍스트 한 줄로 들어옴(D+005 식 파싱 대상)
n_logs = 1200
log_dates = pd.to_datetime("2025-04-01") + pd.to_timedelta(np.random.randint(0, 60, n_logs), unit="D")
ips = [f"{np.random.randint(1,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}" for _ in range(n_logs)]
paths = np.random.choice(["/home", "/product/123", "/cart", "/checkout", "/search?q=shoes"], n_logs)
web_logs = pd.DataFrame({
    "raw_log": [f"{d.strftime('%Y-%m-%d %H:%M:%S')} - {ip} - GET {p}" for d, ip, p in zip(log_dates, ips, paths)]
})

# CSV로 임시 저장(인제스트 단계에서 읽어옴) — 분석가가 "파일을 받았다" 상황 재현
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)
customers.to_csv(RAW_DIR / "customers.csv", index=False)
products.to_csv(RAW_DIR / "products.csv", index=False)
orders.to_csv(RAW_DIR / "orders.csv", index=False)
web_logs.to_csv(RAW_DIR / "web_logs.csv", index=False)

print("종합 프로젝트 데이터 생성 및 CSV 저장 완료")
print(f"  customers: {customers.shape}")
print(f"  products : {products.shape}")
print(f"  orders   : {orders.shape}  (중복·결측·이상치 포함)")
print(f"  web_logs : {web_logs.shape}")
print(f"\n저장 경로: {RAW_DIR.resolve()}")

종합 프로젝트 데이터 생성 및 CSV 저장 완료
  customers: (500, 6)
  products : (60, 3)
  orders   : (5003, 7)  (중복·결측·이상치 포함)
  web_logs : (1200, 1)

저장 경로: C:\Users\USER\ai-data-bootcamp\D009\d009_outputs\raw


## 인제스트(Ingest) — 원본 데이터 받아오기

# 1. 인제스트(Ingest) — 원본 데이터 받아오기

분석가의 첫 동작은 **데이터를 메모리로 읽어오는 일**입니다. 이 단계를 흔히 **인제스트(ingest)** 라고 부릅니다. 어디서 읽어오느냐(파일·DB·API)와 무슨 포맷이냐(CSV·Parquet·JSON)가 이후 모든 단계에 영향을 줍니다.

오늘은 가장 흔한 출발점인 **CSV 파일**부터 시작합니다. 그리고 곧, CSV가 가진 한계를 직접 보게 됩니다. 이 한계가 **Part 5에서 Parquet으로 갈아타는 이유**가 됩니다.

> ❓ **이 파트에서 답할 질문:** "같은 데이터를 한 번 저장했다가 다시 읽었을 뿐인데, 왜 자료형이 사라질까요?"

## 💡 쉽게 말하면 — CSV가 잃어버리는 것

CSV는 사실상 **줄바꿈으로 나눈 문자열 표**입니다. 컴퓨터가 보기에는 모든 값이 그냥 글자입니다.

```text
order_date
2025-01-01     ← 사람 눈엔 '날짜', CSV에겐 그냥 글자 10자
2025/01/01     ← 형식이 달라도 다 똑같은 글자일 뿐
20250101       ← 이것도 마찬가지
```

이 때문에 CSV를 읽으면 pandas는 자료형을 **추측**해야 합니다. 추측이 빗나가면 날짜가 문자열로 들어오고, 큰 정수가 실수로 변하고, 결측치는 빈 칸인지 NaN인지 모호해집니다.

## 자세히 알아보기

| 단계 | 무엇을 하는가 | 오늘 다룰 도구 |
| --- | --- | --- |
| 인제스트(Ingest) | 외부 파일·DB로부터 메모리로 적재 | `pd.read_csv`, `pd.read_parquet` |
| 진단(Diagnose) | 구조·품질을 점검 | 직접 만들 **품질 리포트 함수** |
| 정제(Clean) | 결측·이상·문자열·날짜 처리 | D+003·D+005에서 배운 기법 |
| 변환(Transform) | 병합·집계·파생 | D+004·D+006에서 배운 기법 |
| 저장(Save) | 다음 단계가 쓸 형태로 보관 | **Parquet** (Part 5) |

> 💡 **개념 연결:** D+002에서 `info()`로 자료형을 진단했던 기억을 떠올려보세요. 오늘 우리는 그 진단을 **자동으로 매번 수행하는 함수**를 만듭니다.

## 데이터로 확인해 봅시다 — CSV 읽기와 그 한계

> **읽는 법:** `order_date`가 `object`(문자열)로 들어왔습니다. CSV에 저장하기 전 `customers`의 `signup_date`는 `datetime`이었는데, 다시 읽었더니 어떻게 됐는지 다음 셀에서 확인합니다. *CSV는 자료형을 기억해주지 않습니다.*

> **읽는 법:** 저장 전에는 `datetime64[ns]`였던 컬럼이, CSV에 들어갔다 나오면 `object`가 됩니다. 분석가는 매번 `pd.to_datetime()`을 다시 해줘야 합니다. 이런 식의 **자료형 재추측 비용**이 CSV가 분석 파이프라인의 영구 저장 포맷으로 부적합한 핵심 이유입니다.

> 📌 **다른 산업에서는?** 이 자료형 손실 문제는 금융(거래 시각·소수점 정밀도), 헬스케어(환자 ID가 큰 정수에서 실수로 변형), 제조(센서 시간 스탬프)에서도 동일하게 발생합니다. 그래서 분석 인프라가 성숙한 조직일수록 원본은 그대로 두되 **분석용 저장은 Parquet**으로 통일합니다.

### 스스로 해보자! ✏️ (1)

"정답은 하나가 아닙니다. 일단 실행해보세요."

`orders_csv`의 `amount` 컬럼은 CSV에서 어떤 자료형으로 읽혔나요? 그리고 `quantity`는요? 둘이 다른 이유를 한 문장으로 적어보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
print(orders_csv[["amount", "quantity"]].dtypes)
# amount는 결측치가 섞여 있어 float64로,
# quantity는 결측이 없는 정수라 int64로 잡힙니다.
# 즉, '같은 숫자 컬럼'도 결측 여부에 따라 자료형이 달라집니다.
```

</details>

### ✅ 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. CSV가 자료형을 "기억하지 못한다"는 말은 정확히 무슨 뜻인가요?
2. `int64`였던 컬럼이 결측치를 만나면 어떤 자료형으로 추론되나요?
3. 인제스트 직후 가장 먼저 해야 할 일은 무엇인가요?

> 💡 **다음 Part 예고:** CSV에서 받은 데이터의 자료형이 어긋나 있다면, *그것까지 자동으로 잡아내는 진단 함수*가 있어야 합니다. Part 2에서 직접 만듭니다.

In [3]:
# 예제: 방금 저장한 CSV를 다시 읽어 옵니다. (분석가가 새 데이터를 '받은' 상황 재현)
orders_csv = pd.read_csv(RAW_DIR / "orders.csv")
customers_csv = pd.read_csv(RAW_DIR / "customers.csv")
products_csv = pd.read_csv(RAW_DIR / "products.csv")

print("orders_csv.shape :", orders_csv.shape)
print("customers_csv.shape :", customers_csv.shape)
print("products_csv.shape :", products_csv.shape)
print()
print("orders_csv.dtypes")
print(orders_csv.dtypes)

orders_csv.shape : (5003, 7)
customers_csv.shape : (500, 6)
products_csv.shape : (60, 3)

orders_csv.dtypes
order_id           str
customer_id        str
product_id         str
quantity         int64
amount         float64
channel            str
order_date         str
dtype: object


In [4]:
# 예제: customers의 signup_date가 CSV 왕복 후 어떻게 변했는지 확인
print("CSV에서 읽은 직후 dtypes:")
print(customers_csv[["customer_id", "age", "signup_date"]].dtypes)
print()
print("signup_date 첫 5개 값(문자열로 보임):")
print(customers_csv["signup_date"].head().to_list())
print()
# 같은 컬럼을 datetime으로 다시 파싱해야 사용 가능 — 매번 이 작업을 반복해야 합니다.
parsed = pd.to_datetime(customers_csv["signup_date"])
print("파싱 후 dtype:", parsed.dtype)

CSV에서 읽은 직후 dtypes:
customer_id      str
age            int64
signup_date      str
dtype: object

signup_date 첫 5개 값(문자열로 보임):
['2024-07-18', '2025-05-19', '2024-03-27', '2024-02-12', '2025-01-19']

파싱 후 dtype: datetime64[us]


In [5]:
# 스스로 해보자! (1)
# 1) amount, quantity의 dtype을 출력해보세요.
# 2) 결측치(NaN)가 있는 컬럼은 어떤 자료형으로 추론되는지 관찰해보세요.

# 여기에 코드를 작성하세요

## 데이터 품질 리포트 함수 — 진단을 자동화한다

# 2. 데이터 품질 리포트 함수 — 진단을 자동화한다

D+002에서 우리는 `shape`·`info`·`describe`를 손으로 호출해 데이터를 진찰했습니다. 그러나 종합 프로젝트 파이프라인은 **매번 같은 데이터셋**이 아니라, **매번 새 데이터**가 들어오는 상황에서 돌아야 합니다. 그래서 진단 자체가 **함수**가 되어야 합니다.

> ❓ **이 파트에서 답할 질문:** "결측·중복·이상치·타입 불일치를 한 번에 알려주는 진단 함수를 어떻게 만들까요?"

## 💡 쉽게 말하면 — 자동 건강검진표

병원의 건강검진표는 사람을 다섯 가지 축으로 점검합니다(혈압·혈당·콜레스테롤·체질량·간기능). 항목은 정해져 있고, 누가 와도 같은 양식으로 출력됩니다.

데이터 품질 리포트도 똑같습니다. 다음 네 축을 **함수 하나가 항상 같은 형식으로** 출력합니다.

```text
[1] 구조       행 수 · 열 수 · 메모리
[2] 결측치     컬럼별 결측 개수 · 결측률(%)
[3] 중복       완전 중복 행 수 · 키 컬럼 기준 중복
[4] 타입       각 컬럼의 추론된 dtype + 의심 컬럼
```

## 자세히 알아보기

| 검사 항목 | 기준 | pandas 도구 |
| --- | --- | --- |
| 결측치(missing) | `isna().sum()` 으로 컬럼별 개수, 전체 대비 비율 | `df.isna().mean()` |
| 중복(duplicates) | 완전 중복 행 · 키 컬럼 기준 중복 | `df.duplicated().sum()` |
| 이상치(outliers) | 수치형 컬럼에 IQR(D+003) 적용 | `quantile` |
| 타입 불일치 | object인데 사실은 날짜·숫자인 컬럼 | `pd.to_datetime(errors='coerce')` 시도 |

> 💡 **개념 연결:** D+003에서 IQR로 이상치를 잡았고, D+006에서 `apply`로 함수를 컬럼에 적용했습니다. 그 두 도구가 오늘의 진단 함수 안에 함께 들어갑니다.

## 데이터로 확인해 봅시다 — `quality_report()` 함수 만들기

> **읽는 법:** 한 번 호출에 (1) 전체 크기·메모리, (2) 컬럼별 자료형, (3) 결측 개수와 결측률, (4) 고유값 개수, (5) 실제 값 샘플이 한 표로 나옵니다. *어느 컬럼을 먼저 손볼지 우선순위가 보입니다.* 결측률이 높거나, `n_unique`가 비정상적으로 적은 컬럼이 의심 대상입니다.

> **읽는 법:** 새로 추가된 두 컬럼이 핵심입니다. (1) `outlier_pct_iqr` — 수치형 컬럼에서 IQR 기준 이상치가 몇 %인지 자동 계산. `quantity`나 `amount`에 의심 비율이 보일 겁니다. (2) `maybe_datetime` — `object`로 들어왔지만 사실은 날짜로 보이는 컬럼을 표시. `order_date`가 잡혀야 정상입니다.

> 💡 **개념 연결:** D+003의 IQR 공식, D+005의 `pd.to_datetime(errors='coerce')` 트릭, D+006의 `select_dtypes`가 한 함수 안에서 만났습니다. 종합 프로젝트이 *지금까지 배운 것의 재조합*이라는 뜻이 이 함수 한 개로 분명해집니다.

> **읽는 법:** 같은 함수를 세 번 호출했더니, 세 개의 데이터에 대해 같은 형식의 리포트가 나왔습니다. 이것이 *진단의 표준화* 입니다. 다음 분기, 새 데이터셋이 들어와도 같은 양식으로 곧바로 점검할 수 있습니다.

> 📌 **다른 산업에서는?** 금융에서는 일별 거래 데이터를 받을 때마다 같은 품질 리포트를 자동 출력해 결측·이상이 일정 임계치를 넘으면 알람을 띄웁니다. 헬스케어에서는 EMR(전자 의무기록) 일괄 적재 시 의무 보고 항목입니다.

### 스스로 해보자! ✏️ (2)

"정답은 하나가 아닙니다. 일단 실행해보세요."

`quality_report_full()` 함수에 **컬럼별로 결측률이 30%를 넘으면 ⚠ 표시**가 붙는 컬럼을 하나 더 추가해보세요. 결측률 임계값은 함수의 인자(`missing_threshold=30`)로 받게 만들면 더 좋습니다.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
def quality_report_with_warning(df, name="df", missing_threshold=30.0):
    base = quality_report_full(df, name)
    base["warning"] = np.where(base["missing_pct"] > missing_threshold, "⚠", "")
    return base
```

</details>

### ✅ 짚고 넘어가기

1. `quality_report_full()`이 하는 4가지 진단을 말로 풀어보세요.
2. 왜 IQR이 D+003에서 봤을 때보다 오늘 더 유용하게 느껴지나요? (힌트: 한 줄로 어떻게 변했나요?)
3. 새 데이터셋이 들어와도 이 함수를 **그대로** 쓸 수 있는 이유는?

> 💡 **다음 Part 예고:** 진단을 끝냈으니, 이제 *발견한 문제들*을 실제로 고칠 시간입니다. 결측·이상치·문자열·날짜 정제를 한 줄기로 잇는 Part 3로 갑니다.

In [6]:
# 예제: 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report


# orders에 적용
qr_orders = quality_report(orders_csv, "orders_csv")
qr_orders

[품질 리포트] orders_csv
  행 수: 5,003  /  열 수: 7
  메모리: 1604.5 KB
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,sample
order_id,str,0,0.0,5000,O00001
customer_id,str,0,0.0,500,C0304
product_id,str,0,0.0,60,P053
quantity,int64,0,0.0,4,3
amount,float64,80,1.6,22,89700.0
channel,str,0,0.0,4,app
order_date,str,0,0.0,540,2025-03-08


In [7]:
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


qr_orders_full = quality_report_full(orders_csv, "orders_csv")
qr_orders_full

[품질 리포트(완전판)] orders_csv
  행 수: 5,003  /  열 수: 7
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
order_id,str,0,0.0,5000,NaN,False
customer_id,str,0,0.0,500,NaN,False
product_id,str,0,0.0,60,NaN,False
quantity,int64,0,0.0,4,0.02,False
amount,float64,80,1.6,22,2.11,False
channel,str,0,0.0,4,NaN,False
order_date,str,0,0.0,540,NaN,False


In [8]:
# 예제: customers와 products에도 같은 함수를 적용 — 한 함수가 어떤 데이터든 진단합니다
print("=" * 60)
qr_customers = quality_report_full(customers_csv, "customers_csv")
display(qr_customers)
print("=" * 60)
qr_products = quality_report_full(products_csv, "products_csv")
display(qr_products)

[품질 리포트(완전판)] customers_csv
  행 수: 500  /  열 수: 6
  완전 중복 행: 0건
  📌 날짜로 보이는 object 컬럼: ['signup_date']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
customer_id,str,0,0.0,500,NaN,False
age,int64,0,0.0,54,1.4,False
gender,str,4,0.8,2,NaN,False
region,str,0,0.0,7,NaN,False
membership,str,0,0.0,5,NaN,False
signup_date,str,0,0.0,355,NaN,True


[품질 리포트(완전판)] products_csv
  행 수: 60  /  열 수: 3
  완전 중복 행: 0건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
product_id,str,0,0.0,60,NaN,False
category,str,0,0.0,5,NaN,False
price,int64,0,0.0,7,0.0,False


In [9]:
# 스스로 해보자! (2)
def quality_report_with_warning(df: pd.DataFrame, name: str = "df", missing_threshold: float = 30.0) -> pd.DataFrame:
    '''결측률이 임계치를 넘으면 ⚠ 표시를 붙입니다.'''
    # 여기에 코드를 작성하세요


    pass


# quality_report_with_warning(orders_csv, "orders_csv", missing_threshold=10)

## 정제(Clean) 단계 — 8일간의 정제 기법을 한 줄기로

# 3. 정제(Clean) 단계 — 8일간의 정제 기법을 한 줄기로

품질 리포트가 알려준 문제는 네 종류입니다.

1. **중복**: 완전 중복 행 3건 (`orders` 끝에 심어 둔 것)
2. **결측치**: `amount`에 약 80건, `customers.gender`에 4건
3. **이상치**: `quantity=100`, `age=999`, `amount=-50000`
4. **문자열·날짜 오염**: `channel`의 `app `/`APP`, `region`의 `Seoul`/`서울`, `order_date`의 3가지 포맷

D+003·D+005에서 각각 따로 다뤘던 것들입니다. 오늘은 **순서**를 정해 한 줄기로 잇습니다.

> ❓ **이 파트에서 답할 질문:** "이 네 가지 문제를 어떤 순서로 처리해야 다음 단계가 깨지지 않을까요?"

## 💡 쉽게 말하면 — 청소에도 순서가 있다

방을 청소할 때 카펫을 먼저 청소하고 나서 천장에서 먼지를 떨면 의미가 없습니다. **위에서 아래로**, **굵은 것에서 미세한 것으로** 가는 순서가 있습니다. 데이터 정제도 마찬가지입니다.

```text
[1] 중복 제거       완전 중복부터 빼야 모든 통계가 흔들리지 않음
   ↓
[2] 문자열 정제     'app ', 'APP' 같은 표기를 통일해야 결측·집계가 일관됨
   ↓
[3] 날짜 파싱       문자열 통일 후에 날짜 형식을 datetime으로
   ↓
[4] 이상치 처리     모든 정제 후, 진짜 의심값만 남아 판단이 명료해짐
   ↓
[5] 결측치 처리     마지막에 — 위 단계에서 일부 결측이 자연스레 메워질 수 있음
```

## 자세히 알아보기 — 각 단계에서 쓰는 도구

| 단계 | 핵심 도구 | 출처 |
| --- | --- | --- |
| 중복 제거 | `df.drop_duplicates()` | D+003 |
| 문자열 정제 | `.str.strip()`, `.str.lower()` | D+005 |
| 날짜 파싱 | `pd.to_datetime(format='mixed')` | D+005 |
| 이상치 처리 | IQR · 비즈니스 규칙 | D+003 |
| 결측치 처리 | 채움(`fillna`) · 제거(`dropna`) | D+003 |

> ⚠ **주의:** "어떤 방법이 정답"이 아니라 **"왜 이 방법을 골랐는가"** 가 핵심입니다. 오늘은 각 결정마다 한 줄씩 근거를 코드 옆에 달아둡니다.

## 데이터로 확인해 봅시다 — orders 단계별 정제

> **읽는 법:** 끝에 일부러 심어 둔 3건이 사라졌습니다. 만약 *완전 중복*이 아니라 *키 컬럼(`order_id`) 중복*만 잡고 싶다면 `subset=["order_id"]`를 씁니다. 비즈니스 규칙에 따라 다릅니다.

> **읽는 법:** `app ` (뒤 공백), `APP` (대문자)이 모두 `app`으로 합쳐졌습니다. 이전에 `value_counts`로 4개로 나뉘어 보이던 채널이 이제 2개로 정리됩니다. *집계 결과가 흔들리지 않게 됐습니다.*

> 💡 **개념 연결:** `.str.strip().str.lower()`는 D+005에서 본 메서드 체이닝입니다. 한 줄로 두 정제가 끝나는 우아함이 여기서 빛납니다.

> **읽는 법:** `object` → `datetime64[ns]`로 바뀌면서, 이제 `.dt.year`·`.dt.month`로 시간 컬럼을 자유롭게 다룰 수 있습니다. *Part 4에서 월별 집계를 할 수 있게 된 결정적 순간*입니다.

> 📌 **실무 포인트:** `format='mixed'`는 pandas 2.x부터 지원하는 옵션입니다. 옛 버전이라면 같은 결과를 얻기 위해 컬럼을 여러 번 시도하며 파싱해야 했습니다. 도구의 발전이 곧 작업 분량의 감소입니다.

> **읽는 법:** 음수 금액은 *버리지 않고* `refunds`로 따로 보관했습니다. 이런 결정이 종합 프로젝트의 핵심입니다. *"왜 이 값을 어디로 보냈는지" 추적이 가능*해야 다음 분기의 분석가가 이 노트북을 신뢰할 수 있습니다.

> ⚠ **주의:** `quantity=100` 제거를 1% 컷오프로 잡았지만, *블랙프라이데이 대량 구매* 같은 도메인 맥락이 있다면 다른 판단을 해야 합니다. 코드는 한 줄이지만, 그 뒤의 결정은 비즈니스를 알아야 합니다.

> **읽는 법:** `amount` 결측은 약 80건이었습니다. 행 제거 vs 평균 채움을 두고 *우리 분석 목적이 매출 합산*이라 결측 행을 제거했습니다. 만약 *결제 이상 패턴 분석*이었다면 결측 자체가 정보이므로 살려뒀을 것입니다. **같은 결측, 다른 결정**.

> 💡 **개념 연결:** D+003에서 본 MCAR·MAR·MNAR 구분이 여기서 살아납니다. 우리는 `amount` 결측이 *결제 실패 로그를 못 받아온 시스템 문제*(MCAR에 가깝다)로 가정하고 제거한 것입니다.

### 스스로 해보자! ✏️ (3)

`customers_csv`에서 다음 정제를 순서대로 수행하세요. 각 단계마다 처리 후 `shape`을 출력해 어떻게 변했는지 확인합니다.

1. 완전 중복 제거
2. `region`의 앞뒤 공백 제거 + `'Seoul'` → `'서울'`로 통일
3. `membership`을 소문자로 통일
4. `age`가 0~100 사이가 아니면 결측(NaN)으로 바꾼 뒤 결측 행 제거

<details>
<summary>💡 힌트 (클릭)</summary>

```python
customers_clean = customers_csv.drop_duplicates().copy()
customers_clean["region"] = customers_clean["region"].str.strip().replace({"Seoul": "서울"})
customers_clean["membership"] = customers_clean["membership"].str.strip().str.lower()
customers_clean["age"] = customers_clean["age"].where(customers_clean["age"].between(0, 100))
customers_clean = customers_clean.dropna(subset=["age"]).reset_index(drop=True)
```

`where()`는 조건이 참인 곳만 남기고 거짓인 곳은 NaN으로 바꿉니다. D+003에서 본 결측 만들기와 같은 기법입니다.

</details>

### ✅ 짚고 넘어가기

1. 정제 순서가 *중복 → 문자열 → 날짜 → 이상치 → 결측치* 인 이유를 한 문장으로 설명해보세요.
2. 음수 `amount`를 버리지 않고 `refunds`로 따로 보관한 이유는?
3. "같은 결측, 다른 결정"의 예를 본인 말로 다시 설명해보세요.

> 💡 **다음 Part 예고:** 정제된 데이터를 그대로 두면 *분석가 한 명의 메모장*에 머무릅니다. Part 4에서는 *KPI 표*로 변환합니다. 병합·집계가 합쳐지는 단계입니다.

In [10]:
# 예제 [1단계] 중복 제거 — 가장 먼저
print(f"중복 제거 전: {len(orders_csv):,}행")

# 판단 근거: 완전 동일한 행은 시스템 입력 오류로 간주, 첫 번째만 남깁니다.
orders_step1 = orders_csv.drop_duplicates(keep="first").reset_index(drop=True)

print(f"중복 제거 후: {len(orders_step1):,}행  (제거된 행: {len(orders_csv) - len(orders_step1)}건)")

중복 제거 전: 5,003행
중복 제거 후: 5,000행  (제거된 행: 3건)


In [11]:
# 예제 [2단계] 문자열 정제 — channel, region 표기 통일
print("정제 전 channel 분포:")
print(orders_step1["channel"].value_counts())
print()

# 판단 근거: 앞뒤 공백·대소문자 차이는 같은 값으로 본다 ('APP' → 'app').
orders_step2 = orders_step1.copy()
orders_step2["channel"] = orders_step2["channel"].str.strip().str.lower()

print("정제 후 channel 분포:")
print(orders_step2["channel"].value_counts())

정제 전 channel 분포:
channel
app     2252
web     2251
APP      250
app      247
Name: count, dtype: int64

정제 후 channel 분포:
channel
app    2749
web    2251
Name: count, dtype: int64


In [12]:
# 예제 [3단계] 날짜 파싱 — 3가지 포맷 혼재를 datetime으로 통일
print("정제 전 order_date 자료형:", orders_step2["order_date"].dtype)
print("샘플 값:", orders_step2["order_date"].head(3).to_list())
print()

# 판단 근거: format='mixed'로 자동 추론. 실패하면 NaT(결측 시각)로 표시.
orders_step3 = orders_step2.copy()
orders_step3["order_date"] = pd.to_datetime(
    orders_step3["order_date"], format="mixed", errors="coerce"
)

print("정제 후 order_date 자료형:", orders_step3["order_date"].dtype)
print("NaT(파싱 실패) 건수:", orders_step3["order_date"].isna().sum())
print("샘플 값:", orders_step3["order_date"].head(3).to_list())

정제 전 order_date 자료형: str
샘플 값: ['2025-03-08', '2025/04/21', '20250608']

정제 후 order_date 자료형: datetime64[us]
NaT(파싱 실패) 건수: 0
샘플 값: [Timestamp('2025-03-08 00:00:00'), Timestamp('2025-04-21 00:00:00'), Timestamp('2025-06-08 00:00:00')]


In [13]:
# 예제 [4단계] 이상치 처리 — 비즈니스 규칙 + IQR 혼합
print("이상치 처리 전 amount/quantity 통계:")
print(orders_step3[["amount", "quantity"]].describe().round(0))
print()

# 판단 근거 1: 음수 금액은 환불일 수도 있으나, 이번 분석은 '구매'를 보는 것이므로 별도 보관 후 본 테이블에선 제외.
orders_step4 = orders_step3.copy()
refunds = orders_step4[orders_step4["amount"] < 0].copy()
orders_step4 = orders_step4[(orders_step4["amount"] >= 0) | orders_step4["amount"].isna()]

# 판단 근거 2: quantity=100은 단일 거래로 비현실적. 99 percentile 기준으로 자릅니다.
q99 = orders_step4["quantity"].quantile(0.99)
orders_step4 = orders_step4[orders_step4["quantity"] <= q99]

print(f"음수 금액 분리: {len(refunds)}건 (별도 보관)")
print(f"수량 상위 1% 컷오프: {q99}")
print(f"이상치 처리 후 행 수: {len(orders_step4):,}")
print()
print("이상치 처리 후 quantity 통계:")
print(orders_step4["quantity"].describe().round(2))

이상치 처리 전 amount/quantity 통계:
         amount  quantity
count    4920.0    5000.0
mean   136747.0       2.0
std    128622.0       2.0
min    -50000.0       1.0
25%     29900.0       1.0
50%     99800.0       2.0
75%    199900.0       2.0
max    599700.0     100.0

음수 금액 분리: 1건 (별도 보관)
수량 상위 1% 컷오프: 3.0
이상치 처리 후 행 수: 4,998

이상치 처리 후 quantity 통계:
count    4998.00
mean        1.67
std         0.74
min         1.00
25%         1.00
50%         2.00
75%         2.00
max         3.00
Name: quantity, dtype: float64


In [14]:
# 예제 [5단계] 결측치 처리 — 마지막에 처리
print("결측치 처리 전:")
print(orders_step4.isna().sum())
print()

orders_clean = orders_step4.copy()

# 판단 근거: amount 결측은 '집계 왜곡 방지'를 위해 행 제거. (대안: 컬럼 평균으로 채움 — 분석 목적에 따라 다름)
orders_clean = orders_clean.dropna(subset=["amount"])

print("결측치 처리 후:")
print(orders_clean.isna().sum())
print()
print(f"최종 정제된 orders: {len(orders_clean):,}행")

결측치 처리 전:
order_id        0
customer_id     0
product_id      0
quantity        0
amount         80
channel         0
order_date      0
dtype: int64

결측치 처리 후:
order_id       0
customer_id    0
product_id     0
quantity       0
amount         0
channel        0
order_date     0
dtype: int64

최종 정제된 orders: 4,918행


In [15]:
# 스스로 해보자! (3)
customers_clean = customers_csv.copy()
# 여기에 코드를 작성하세요


# print(customers_clean.shape)

## 변환(Transform) 단계 — 분석 가능한 형태로

# 4. 변환(Transform) 단계 — 분석 가능한 형태로

정제된 `orders_clean`은 깨끗하지만, *그 자체로는 인사이트가 없습니다*. 분석 가능한 형태란 보통 **요약 테이블**(월별 매출, 고객 세그먼트별 평균 등)입니다. 변환 단계의 목표는 두 가지입니다.

1. 흩어진 테이블 합치기 (D+004의 `merge`)
2. 집계해서 KPI 표 만들기 (D+004의 `groupby`)

> ❓ **이 파트에서 답할 질문:** "정제된 데이터를 어떤 형태로 변환해야 다음 분기 분석가가 바로 쓸 수 있을까요?"

## 💡 쉽게 말하면 — '청구서' 만들기

영수증 한 장은 한 번의 거래입니다. 청구서는 한 달의 거래를 *고객별·상품별로* 묶어 한 장에 정리한 것입니다. 변환 단계는 **영수증 더미를 청구서로 바꾸는 일**입니다.

```text
[영수증] 한 줄 = 한 주문                    (raw orders)
   ↓ merge: 고객 정보·상품 정보 붙이기
[보강된 영수증] 한 줄 = 한 주문 + 누구·무엇  (joined)
   ↓ groupby: 묶어서 합치기
[청구서] 한 줄 = 한 (월·세그먼트) KPI       (summary)
```

## 자세히 알아보기

| 변환 종류 | 도구 | 결과 |
| --- | --- | --- |
| 테이블 결합 | `pd.merge` (`how='left'`) | 행 수 유지, 컬럼 추가 |
| 집계 | `groupby().agg()` | 행 수 축소, 요약값 |
| 파생 컬럼 | 단순 산술 + `assign` | 비율·플래그 등 새 컬럼 |
| 형태 변경 | `pivot_table` / `melt` | wide ↔ long |

> 💡 **개념 연결:** D+006에서 `.assign()`이 method chaining의 핵심이라고 했습니다. 이번 Part에서 그 가치를 끝까지 보게 됩니다.

## 데이터로 확인해 봅시다 — 월별·카테고리별 KPI 표 만들기

> **읽는 법:** `merge`를 두 번 연달아 하니, `orders`의 각 행에 *누가 무엇을 샀는지*가 모두 붙었습니다. 이제 `region` 기준 집계, `category` 기준 집계가 자유롭습니다. `how='left'`를 쓴 덕에 *주문 건수는 변하지 않았습니다.* (inner였다면 매칭 실패한 행이 사라졌을 것입니다.)

> **읽는 법:** `agg()` 안에 *별명 = (컬럼, 함수)* 형식을 쓰면 결과 컬럼 이름을 바로 정할 수 있습니다. *분석가가 다음 분기에 받는 KPI 표는 거의 항상 이 모양입니다.*

> 📌 **다른 산업에서는?** 같은 패턴이 마케팅(채널별 CTR), 금융(상품별 NPL률), 헬스케어(과목별 외래 환자수)에도 똑같이 적용됩니다. 컬럼 이름만 바뀝니다.

> **읽는 법:** `groupby` 후 `pivot_table`로 long → wide 변환했습니다. 사람 눈에는 wide가 한눈에 비교하기 좋고, 컴퓨터(다른 분석 코드)에는 long이 다루기 좋습니다. *목적에 따라 형태를 바꿉니다.* — D+004에서 봤던 wide/long 변환의 실전 활용입니다.

### 스스로 해보자! ✏️ (4)

`merged_with_month`를 이용해 다음을 구해보세요.

- `region`(지역)별 월별 매출(`total_revenue`)을 wide 표로 만들기 (행=region, 열=월)
- 그 표에서 *가장 매출이 큰 (지역, 월) 조합* 3개를 출력

<details>
<summary>💡 힌트 (클릭)</summary>

```python
region_month = (
    merged_with_month
    .groupby(["region", "order_month"])["amount"].sum()
    .unstack(fill_value=0)
    .round(0)
)
print(region_month)

# 상위 3 조합
top3 = (
    merged_with_month.groupby(["region", "order_month"])["amount"]
    .sum().nlargest(3)
)
print(top3)
```

</details>

### ✅ 짚고 넘어가기

1. `merge`에서 `how='left'`를 쓴 이유는 무엇인가요?
2. `agg(별명=(컬럼, 함수))` 패턴의 장점은?
3. wide 표 vs long 표 — 각각 언제 쓰면 좋을까요?

> 💡 **다음 Part 예고:** 변환까지 끝났습니다. 이제 다음 분기 분석가가 받아볼 수 있는 **결과 파일**로 저장합니다. 왜 CSV가 아니라 Parquet인지 — Part 5에서 직접 비교합니다.

In [16]:
# 예제 [1] 테이블 결합 — orders + customers + products
# 판단 근거: orders 기준으로 모두 left join. orders에 매칭 안 되는 고객/상품은 어차피 매출 표에 안 잡힘.

# customers와 products는 Part 3 스타일로 빠르게 정제
customers_clean2 = (
    customers_csv.drop_duplicates()
    .assign(
        region=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
        membership=lambda d: d["membership"].str.strip().str.lower(),
        gender=lambda d: d["gender"].fillna("Unknown"),
    )
    .pipe(lambda d: d[d["age"].between(0, 100)])
    .reset_index(drop=True)
)

merged = (
    orders_clean
    .merge(customers_clean2[["customer_id", "region", "membership", "age"]], on="customer_id", how="left")
    .merge(products[["product_id", "category", "price"]], on="product_id", how="left")
)

print("결합 후 shape:", merged.shape)
print("결합 후 컬럼:", list(merged.columns))
merged.head(3)

결합 후 shape: (4918, 12)
결합 후 컬럼: ['order_id', 'customer_id', 'product_id', 'quantity', 'amount', 'channel', 'order_date', 'region', 'membership', 'age', 'category', 'price']


,order_id,customer_id,product_id,quantity,amount,channel,order_date,region,membership,age,category,price
0,O00001,C0304,P053,3,89700.0,app,2025-03-08,경기,basic,42.0,패션,29900
1,O00002,C0276,P016,1,129900.0,web,2025-04-21,부산,basic,41.0,가전,129900
2,O00003,C0357,P044,2,19800.0,app,2025-06-08,경기,basic,36.0,가전,9900


In [17]:
# 예제 [2] 집계 — 월별 매출 KPI + 카테고리별 객단가
merged_with_month = merged.assign(
    order_month=lambda d: d["order_date"].dt.to_period("M").astype(str)
)

# 월별 KPI
monthly_kpi = (
    merged_with_month
    .groupby("order_month")
    .agg(
        total_revenue=("amount", "sum"),
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        avg_order_value=("amount", "mean"),
    )
    .round(0)
    .reset_index()
)

print("월별 KPI:")
display(monthly_kpi)

월별 KPI:


,order_month,total_revenue,order_count,unique_customers,avg_order_value
0,2025-01,110338200.0,810,393,136220.0
1,2025-02,104596200.0,801,395,130582.0
2,2025-03,114529400.0,843,411,135859.0
3,2025-04,115705000.0,847,416,136606.0
4,2025-05,117985500.0,853,412,138318.0
5,2025-06,109432700.0,764,389,143237.0


In [18]:
# 예제 [3] 파생 컬럼 + 피벗 — 세그먼트별 평균 객단가를 wide 표로
segment_aov = (
    merged_with_month
    .groupby(["membership", "category"])
    .agg(aov=("amount", "mean"))
    .round(0)
    .reset_index()
    .pivot_table(index="membership", columns="category", values="aov")
    .round(0)
)

print("세그먼트(membership) x 카테고리 평균 객단가:")
segment_aov

세그먼트(membership) x 카테고리 평균 객단가:


category,가전,도서,뷰티,식품,패션
membership,,,,,
basic,133922.0,145756.0,115037.0,161290.0,111632.0
premium,120858.0,153347.0,122144.0,166303.0,103757.0
vip,139596.0,183534.0,135818.0,149930.0,106718.0


In [19]:
# 스스로 해보자! (4)
# 여기에 코드를 작성하세요

## 저장(Save) 단계 — Parquet vs CSV

# 5. 저장(Save) 단계 — Parquet vs CSV

Part 1에서 우리는 CSV가 **자료형을 기억하지 못한다**는 한계를 보았습니다. 정제와 변환에 그토록 공을 들여 만든 결과를 *다시 CSV로 저장*하면, 다음 분기 분석가는 다시 `pd.to_datetime`을 호출해야 합니다. 그 손실을 막는 포맷이 **Parquet**입니다.

> ❓ **이 파트에서 답할 질문:** "같은 데이터를 CSV와 Parquet으로 저장하면 무엇이 어떻게 다를까요?"

## 💡 쉽게 말하면 — 사진과 PSD 파일

```text
JPG (사진)        — 사람이 바로 보기엔 편함. 그러나 레이어·자료형은 사라짐.
PSD (포토샵)      — 레이어·텍스트·필터를 다 보존. 후속 작업이 가능.
```

CSV는 JPG에, Parquet은 PSD에 가깝습니다. CSV는 모든 도구가 읽을 수 있어 *유통이 편한* 대신 *구조가 단순해서 자료형이 사라*집니다. Parquet은 *분석 도구 간 표준*으로, 자료형·압축·컬럼 단위 읽기가 가능합니다.

## 자세히 알아보기

| 항목 | CSV | Parquet |
| --- | --- | --- |
| 저장 방식 | 행 단위 텍스트 | 컬럼 단위 바이너리 |
| 자료형 보존 | ❌ (재추측) | ✅ (스키마 저장) |
| 압축 | 없음 (또는 외부 zip) | 내장 (snappy/gzip) |
| 파일 크기 | 큼 | 보통 1/5 ~ 1/10 |
| 컬럼 일부만 읽기 | 불가능 | 가능 (`columns=[...]`) |
| 사람이 텍스트 에디터로 열기 | ✅ | ❌ (바이너리) |

> 💡 **개념 연결:** D+007에서 본 Polars도 Parquet과 궁합이 좋습니다. *대용량을 다룰수록 Parquet의 이득이 커집니다.*

## 데이터로 확인해 봅시다 — 같은 데이터 두 포맷으로 저장하고 비교

> **읽는 법:** 보통 Parquet 파일이 CSV의 *1/5에서 1/10* 정도 됩니다. 컬럼 단위 압축 덕입니다. 데이터가 클수록 이 차이는 *디스크 비용*과 *전송 시간*에서 그대로 절약으로 이어집니다.

> **읽는 법:** 같은 컬럼(`order_date`)을 두 포맷에서 다시 읽었을 때 **자료형이 다릅니다**. CSV는 `object`(문자열)로, Parquet은 `datetime64[ns]`로 돌아왔습니다. *우리가 Part 3에서 공들여 파싱한 자료형이 Parquet에서는 그대로 살아 있습니다.* 이것이 핵심 차이입니다.

> **읽는 법:** CSV는 *파일 전체를 다 읽고 나서야* 컬럼을 고를 수 있습니다. Parquet은 *필요한 컬럼만 디스크에서 꺼냅니다*. 대용량(수 GB)에서는 메모리·시간 모두에서 큰 차이가 납니다.

> 📌 **다른 산업에서는?** 대규모 광고 로그(마케팅), 거래 원장(금융), 센서 로그(제조)에서는 Parquet이 사실상 표준입니다. AWS S3 위의 분석 파이프라인, Spark, DuckDB, Polars 모두 Parquet을 1급 시민으로 다룹니다.

### 스스로 해보자! ✏️ (5)

`monthly_kpi`를 CSV와 Parquet으로 각각 저장하고, 두 파일 크기를 출력해보세요. 그리고 Parquet으로 저장된 파일에서 `["order_month", "total_revenue"]` 두 컬럼만 다시 읽어보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
csv_p = OUT_DIR / "monthly_kpi.csv"
pq_p = OUT_DIR / "monthly_kpi.parquet"

monthly_kpi.to_csv(csv_p, index=False)
monthly_kpi.to_parquet(pq_p, index=False)

print("CSV     :", csv_p.stat().st_size, "bytes")
print("Parquet :", pq_p.stat().st_size, "bytes")

partial = pd.read_parquet(pq_p, columns=["order_month", "total_revenue"])
print(partial)
```

</details>

### ✅ 짚고 넘어가기

1. Parquet이 CSV보다 보통 더 작은 이유 두 가지를 들어보세요. (컬럼 저장 / 압축)
2. "자료형 보존"이 왜 분석 파이프라인에서 중요한가요?
3. CSV로 저장해야 하는 상황은 어떤 때인가요? (힌트: 사람이 직접 열어볼 때)

> 💡 **다음 Part 예고:** 이제 1~5단계가 모두 **함수와 줄글**로 흩어져 있습니다. Part 6에서 `.pipe()`로 묶어 하나의 파이프라인 함수로 만듭니다.

In [20]:
# 예제: 같은 데이터를 CSV / Parquet으로 각각 저장하고 용량·속도·자료형을 비교

# 저장 대상: 정제·변환까지 끝난 merged_with_month
csv_path = OUT_DIR / "merged_with_month.csv"
parquet_path = OUT_DIR / "merged_with_month.parquet"

# 1) 저장 속도 측정
t0 = time.time()
merged_with_month.to_csv(csv_path, index=False)
csv_write = time.time() - t0

t0 = time.time()
merged_with_month.to_parquet(parquet_path, index=False)   # pyarrow 사용
pq_write = time.time() - t0

# 2) 용량 측정
csv_size = csv_path.stat().st_size / 1024
pq_size = parquet_path.stat().st_size / 1024

print(f"[저장 속도]  CSV: {csv_write*1000:.1f}ms  /  Parquet: {pq_write*1000:.1f}ms")
print(f"[파일 용량]  CSV: {csv_size:>7.1f} KB  /  Parquet: {pq_size:>7.1f} KB  (Parquet은 CSV의 {pq_size/csv_size*100:.1f}%)")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
# 예제: 읽기 속도와 자료형 보존을 비교

t0 = time.time()
from_csv = pd.read_csv(csv_path)
csv_read = time.time() - t0

t0 = time.time()
from_parquet = pd.read_parquet(parquet_path)
pq_read = time.time() - t0

print(f"[읽기 속도]  CSV: {csv_read*1000:.1f}ms  /  Parquet: {pq_read*1000:.1f}ms")
print()
print("CSV에서 다시 읽은 order_date dtype:", from_csv["order_date"].dtype)
print("Parquet에서 다시 읽은 order_date dtype:", from_parquet["order_date"].dtype)

In [ ]:
# 예제: Parquet의 추가 매력 — 컬럼 일부만 빠르게 읽기
# 판단 근거: 다음 분기 분석가는 KPI만 보고 싶을 수 있습니다. 그럴 때 컬럼만 골라 읽으면 빠릅니다.

cols_only = pd.read_parquet(parquet_path, columns=["order_month", "amount", "category"])
print("필요한 3개 컬럼만 읽기:", cols_only.shape)
print(cols_only.head(3))

In [ ]:
# 스스로 해보자! (5)
# 여기에 코드를 작성하세요

## 파이프라인 함수화 — `.pipe()`로 한 흐름

# 6. 파이프라인 함수화 — `.pipe()`로 한 흐름

Part 3~5에서 우리는 *결정 하나마다 셀 하나*를 썼습니다. 학습용으로는 적합하지만, **재사용**과 **자동화**에는 부적합합니다. 다음 분기에는 셀을 또 줄줄이 실행하는 게 아니라, *함수 한 번 호출*로 끝나야 합니다.

> ❓ **이 파트에서 답할 질문:** "흩어진 단계를 어떻게 한 줄짜리 호출로 묶을까요? 그리고 어떻게 *읽기 쉽게* 유지할까요?"

## 💡 쉽게 말하면 — 컨베이어 벨트

```text
[원본 DF]
   │  step1: 중복 제거
   ▼
[중복 제거됨]
   │  step2: 문자열 정제
   ▼
[표기 통일]
   │  step3: 날짜 파싱
   ▼
[날짜 정상]
   │  step4: 이상치 처리
   ▼
[이상치 제거]
   │  step5: 결측치 처리
   ▼
[정제 완료]
```

각 단계가 **DF를 받아 DF를 돌려주는 함수**가 되면, 이 컨베이어 벨트를 코드로 그릴 수 있습니다. 그 도구가 `.pipe()`입니다.

## 자세히 알아보기

| 패턴 | 코드 모양 | 가독성 |
| --- | --- | --- |
| 중첩 호출 | `step5(step4(step3(step2(step1(df)))))` | ❌ 안에서 밖으로 읽어야 함 |
| 변수 누적 | `df1 = step1(df0); df2 = step2(df1); ...` | △ 중간 변수가 많아짐 |
| **`.pipe()`** | `df.pipe(step1).pipe(step2).pipe(step3)` | ✅ 위에서 아래로 흐름이 읽힘 |

> 💡 **개념 연결:** D+006에서 본 method chaining의 완성형입니다. `.pipe()`는 *내가 만든 함수*를 메서드처럼 줄에 끼워 넣어 줍니다.

## 데이터로 확인해 봅시다 — 정제 파이프라인을 한 함수로

> **읽는 법:** 각 함수는 *DataFrame을 받아 DataFrame을 돌려줍니다*. 입출력이 동일하기 때문에 어떤 순서로든 끼워 넣을 수 있습니다. **함수의 입출력을 통일하는 것**이 파이프라인 설계의 핵심입니다.

> **읽는 법:** 한 줄짜리 *세로 흐름*이 곧 정제 과정 전체입니다. Part 3에서 *5개의 셀*에 흩어져 있던 것이 *5줄의 `.pipe()` 호출*로 모였습니다. 다음 분기에는 이 함수 하나만 호출하면 됩니다.

> 📌 **실무 포인트:** 각 단계 함수에 **로그**(`print(f"[step] {df.shape}")`)나 **검증**(`assert df['amount'].notna().all()`)을 끼워 넣으면 운영용 파이프라인이 됩니다. 학습용에서는 빼두지만, 실무에서는 거의 항상 들어갑니다.

> **읽는 법:** 인제스트부터 저장까지가 **한 함수 호출**로 끝났습니다. `log` 딕셔너리에는 각 단계의 결과 행 수와 저장 경로가 들어 있어 *다음 분기에 이 함수를 돌렸을 때 무엇이 변했는지*를 추적할 수 있습니다. 운영 단계에서는 이 `log`를 그대로 슬랙·이메일에 보내는 알람 시스템에 연결합니다.

> 💡 **개념 연결:** D+006에서 본 `.pipe()`가 *단계별 정제*를, 그리고 그 단계들을 모은 `full_pipeline()`이 *전체 시스템 함수*가 됩니다. 작은 함수들이 모여 큰 시스템을 만드는 패턴이 *함수형 데이터 엔지니어링*입니다.

## 8일을 모두 모아보기 — Polars 모드와 스케일링 분기 (선택)

지금까지 만든 `full_pipeline()`은 **pandas + `.pipe()`** 조합입니다. 그런데 D+007에서 본 **Polars**나 D+006의 **수치 스케일링**(scaler)이 필요한 상황도 있죠 — 데이터가 *커지거나*, *모델 입력*으로 넘길 때입니다. 8일간 배운 도구를 종합 프로젝트 파이프라인에 *선택적 분기*로 끼워 넣으면 어떻게 보이는지 짧게 맛봅니다.

> 💡 두 분기 모두 *기본 파이프라인을 대체*하지 않습니다. **"필요할 때만 켜는 모드"** 로 두는 게 운영 노트북의 정석이에요.

> **읽는 법:** **Polars 분기**는 *대용량 인제스트*를 위해, **scaler 분기**는 *모델 입력 준비*를 위해 켭니다. 두 분기 모두 *전체 파이프라인 안의 한 노드*로 들어가는 게 자연스러워요. 가령 인제스트 함수만 Polars로, 변환 후 한 단계만 scaler로 바꿔도 됩니다.
>
> 💡 **개념 연결:** D+006에서 *함수로 묶고*, D+007에서 *도구를 고르는 법*을 배웠고, 오늘 그 둘이 같은 파이프라인 안에서 *분기 선택*으로 만난다는 점이 종합 프로젝트의 진짜 통합입니다.

### 스스로 해보자! ✏️ (6)

`full_pipeline()`을 수정해서 다음을 추가해보세요.

1. 정제 *전후*의 `orders` 행 수 변화를 `log["row_change"]`에 기록
2. 변환 결과의 **월별 매출 총합**을 계산해 `log["monthly_revenue"]`에 dict 형태로 저장

기존 함수를 통째로 복사한 뒤 두 줄만 추가하면 됩니다.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
def full_pipeline_v2(input_dir, output_path):
    log = {}
    orders = pd.read_csv(input_dir / "orders.csv")
    customers = pd.read_csv(input_dir / "customers.csv")
    products = pd.read_csv(input_dir / "products.csv")

    orders_c = (orders.pipe(drop_duplicates_step).pipe(clean_channel_step)
                .pipe(parse_date_step).pipe(remove_outliers_step)
                .pipe(drop_missing_amount_step))

    log["row_change"] = {"before": len(orders), "after": len(orders_c)}

    merged = (
        orders_c
        .merge(customers, on="customer_id", how="left")
        .merge(products[["product_id", "category"]], on="product_id", how="left")
        .assign(order_month=lambda d: d["order_date"].dt.to_period("M").astype(str))
    )
    log["monthly_revenue"] = merged.groupby("order_month")["amount"].sum().round(0).to_dict()

    merged.to_parquet(output_path, index=False)
    return log
```

</details>

### ✅ 짚고 넘어가기

1. 왜 각 정제 단계를 *작은 함수*로 떼어냈을까요?
2. `.pipe()`와 중첩 호출(`step3(step2(step1(df)))`)의 가독성 차이를 한 문장으로 설명해보세요.
3. `log` 딕셔너리는 *코드 외에* 어떤 기능을 제공하나요?

> 💡 **다음 Part 예고:** 마지막으로 *판단 근거*를 코드 옆에 글로 남기는 법을 정리합니다. 동료가 이 노트북을 읽을 때 가장 먼저 보는 부분입니다.

In [ ]:
# 예제: 각 정제 단계를 작은 함수로 떼어내기 — '한 함수 = 한 책임' 원칙
def drop_duplicates_step(df):
    '''1단계: 완전 중복 행 제거.'''
    return df.drop_duplicates(keep="first").reset_index(drop=True)


def clean_channel_step(df):
    '''2단계: channel의 공백·대소문자 정제.'''
    return df.assign(channel=df["channel"].str.strip().str.lower())


def parse_date_step(df):
    '''3단계: order_date 문자열을 datetime으로 파싱.'''
    return df.assign(
        order_date=pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    )


def remove_outliers_step(df):
    '''4단계: 음수 금액 제거 + 수량 상위 1% 컷오프.'''
    q99 = df["quantity"].quantile(0.99)
    cond = ((df["amount"] >= 0) | df["amount"].isna()) & (df["quantity"] <= q99)
    return df[cond]


def drop_missing_amount_step(df):
    '''5단계: amount 결측 행 제거.'''
    return df.dropna(subset=["amount"]).reset_index(drop=True)


print("5개의 단계 함수가 정의되었습니다. 다음 셀에서 .pipe()로 묶습니다.")

In [ ]:
# 예제: .pipe()로 5단계를 한 흐름으로 묶기 + 시간 측정
t0 = time.time()
orders_clean_v2 = (
    orders_csv
    .pipe(drop_duplicates_step)
    .pipe(clean_channel_step)
    .pipe(parse_date_step)
    .pipe(remove_outliers_step)
    .pipe(drop_missing_amount_step)
)
elapsed = time.time() - t0

print(f"파이프라인 실행: {elapsed*1000:.1f}ms")
print(f"최종 shape: {orders_clean_v2.shape}")
print()
print("이전 셀 단위 결과(orders_clean)와 동일한가?",
      orders_clean_v2.shape == orders_clean.shape)

In [ ]:
# 예제: 인제스트~저장까지 전체 파이프라인을 하나의 함수로
def full_pipeline(input_dir: Path, output_path: Path) -> dict:
    '''종합 프로젝트 end-to-end: 인제스트 → 진단 → 정제 → 변환 → 저장.

    돌려주는 dict에는 단계별 행 수, 저장 경로, 보존된 환불 데이터가 들어갑니다.
    '''
    log = {}

    # 1) 인제스트
    orders = pd.read_csv(input_dir / "orders.csv")
    customers = pd.read_csv(input_dir / "customers.csv")
    products = pd.read_csv(input_dir / "products.csv")
    log["ingest_rows"] = {"orders": len(orders), "customers": len(customers), "products": len(products)}

    # 2) 진단
    log["quality_before"] = orders.isna().sum().to_dict()

    # 3) 정제 (위에서 정의한 단계 함수 재사용)
    orders_c = (
        orders
        .pipe(drop_duplicates_step)
        .pipe(clean_channel_step)
        .pipe(parse_date_step)
        .pipe(remove_outliers_step)
        .pipe(drop_missing_amount_step)
    )
    customers_c = (
        customers.drop_duplicates()
        .assign(
            region=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
            membership=lambda d: d["membership"].str.strip().str.lower(),
            gender=lambda d: d["gender"].fillna("Unknown"),
        )
        .pipe(lambda d: d[d["age"].between(0, 100)])
        .reset_index(drop=True)
    )

    # 4) 변환
    merged = (
        orders_c
        .merge(customers_c[["customer_id", "region", "membership", "age"]], on="customer_id", how="left")
        .merge(products[["product_id", "category", "price"]], on="product_id", how="left")
        .assign(order_month=lambda d: d["order_date"].dt.to_period("M").astype(str))
    )

    # 5) 저장
    merged.to_parquet(output_path, index=False)
    log["saved_to"] = str(output_path)
    log["final_rows"] = len(merged)
    return log


# 함수 실행
pipeline_log = full_pipeline(RAW_DIR, OUT_DIR / "pipeline_result.parquet")
for k, v in pipeline_log.items():
    print(f"{k}: {v}")

In [ ]:
# 분기 ① Polars 모드 — 같은 정제 흐름을 Lazy로 표현 (D+007 도구 재활용)
# 오늘 데이터는 작아 속도 이득은 미미하지만, '같은 흐름이 다른 도구로 표현됨'을 확인합니다.
try:
    import polars as pl

    def polars_pipeline_demo(csv_path):
        '''Polars Lazy 파이프라인 — pandas .pipe()와 같은 정제를 표현만 다르게.'''
        return (
            pl.scan_csv(csv_path)                                   # lazy: 아직 안 읽음
              .unique()                                              # 중복 제거
              .with_columns(
                  pl.col("channel").str.strip_chars().str.to_lowercase(),
              )
              .filter(
                  (pl.col("amount") >= 0) | pl.col("amount").is_null()
              )
              .drop_nulls(subset=["amount"])
              .collect()                                              # eager: 이때 실제 실행
        )

    orders_polars = polars_pipeline_demo(RAW_DIR / "orders.csv")
    print(f"Polars 결과 shape: {orders_polars.shape}")
    print("(merge·집계 전 단계라 pandas 최종본과 행 수가 다를 수 있어요 —")
    print(" 핵심은 같은 정제 흐름의 *다른 표현*입니다.)")
except ImportError:
    print("polars 미설치 — 'pip install polars'로 설치하면 이 분기를 켤 수 있습니다.")

In [ ]:
# 분기 ② 수치 스케일링 — 정제 결과를 모델 입력용으로 표준화 (D+006 scaler 재활용)
from sklearn.preprocessing import StandardScaler

merged_loaded = pd.read_parquet(OUT_DIR / "pipeline_result.parquet")
num_cols = [c for c in ["amount", "quantity", "price", "age"] if c in merged_loaded.columns]

scaler = StandardScaler()
scaled_in = merged_loaded[num_cols].fillna(merged_loaded[num_cols].median(numeric_only=True))
scaled_arr = scaler.fit_transform(scaled_in)
scaled_df = pd.DataFrame(scaled_arr, columns=num_cols)

print("스케일링 전 (mean/std):")
print(merged_loaded[num_cols].describe().loc[["mean", "std"]].round(2))
print("\n스케일링 후 (mean≈0, std≈1):")
print(scaled_df.describe().loc[["mean", "std"]].round(2))

In [ ]:
# 스스로 해보자! (6)
def full_pipeline_v2(input_dir: Path, output_path: Path) -> dict:
    log = {}
    # 여기에 코드를 작성하세요


    return log


# full_pipeline_v2(RAW_DIR, OUT_DIR / "pipeline_v2.parquet")

## 판단 근거 문서화 — 동료가 읽을 수 있는 노트북

# 7. 판단 근거 문서화 — 동료가 읽을 수 있는 노트북

지금까지 셀마다 *"판단 근거"* 라고 적은 주석을 봤을 겁니다. 그 줄들이 모이면 한 편의 **결정 로그**가 됩니다. 코드는 *무엇을 했는지*를 보여주지만, **글은 *왜* 했는지를** 보여줍니다. 둘이 다 있어야 노트북이 *문서*가 됩니다.

> ❓ **이 파트에서 답할 질문:** "다음 분기의 분석가가 이 노트북을 처음 열 때, 무엇이 가장 먼저 보여야 할까요?"

## 💡 쉽게 말하면 — '회의록' 같은 노트북

분석 노트북은 **혼자만의 메모장**이 아니라 **회의록**입니다. 결정·근거·반대 의견·다음 액션이 함께 적혀 있어야 합니다. 코드가 의사록이라면, 마크다운 셀은 회의록의 *발언*입니다.

## 자세히 알아보기 — 문서화의 다섯 자리

| 위치 | 무엇을 적는가 | 예시 |
| --- | --- | --- |
| 1. 노트북 상단 | 분석 목적 · 데이터 출처 · 작성자 · 작성일 | "2025 Q2 매출 정제 — 작성자 ○○○" |
| 2. 각 단계 시작 | "이 단계의 목적" 한 줄 | "이상치 처리: IQR + 비즈니스 규칙" |
| 3. 결정 직전 | "판단 근거" 줄 | "음수 amount는 환불로 보고 별도 보관" |
| 4. 결정 직후 | "처리 결과" 줄 | "환불 데이터 N건 분리, 본 분석에서 제외" |
| 5. 노트북 하단 | 한계·후속 작업 | "rainfall 결측 100건은 향후 모델링 시 재검토" |

> 💡 **개념 연결:** 이건 D+006에서 강조한 "팀원이 맥락 없이도 이해하는 코드 작성 습관"의 *글 버전*입니다.

## 데이터로 확인해 봅시다 — 결정 로그 표 자동 생성

> **읽는 법:** 5단계의 결정이 *표 한 장*에 모였습니다. 다음 분기 분석가는 *이 표 한 장만 읽어도* 정제 흐름을 90% 이해합니다. 코드는 그 다음에 봅니다. **표 → 코드** 순으로 읽히는 노트북이 *동료가 읽기 좋은 노트북* 입니다.

> **읽는 법:** 위 출력을 그대로 마크다운 셀에 붙이면 노트북 끝에 *결정 로그* 섹션이 생깁니다. GitHub의 PR에서도 그대로 렌더링됩니다.

> 📌 **실무 포인트:** *결정* 컬럼만 따로 모은 [Architecture Decision Record(ADR)](https://adr.github.io/) 양식을 쓰는 팀도 있습니다. 형식이 무엇이든 핵심은 같습니다 — **결정을 남겨라.**

# 🧪 종합 실습 (종합 프로젝트) — 새 오염 데이터셋, 처음부터 끝까지

지금까지 모두마켓 데이터로 흐름을 따라왔습니다. 이번에는 **새 데이터**를 받았다고 가정합니다. 동료 가게 *"빵셀러"* (베이커리 체인)의 3개월치 운영 데이터입니다. 의도적으로 다양한 오염이 섞여 있습니다.

세 시나리오를 차례로 해결한 뒤 마지막에 **종합 프로젝트 보고서**(마크다운)를 채우세요. 모범 답안 코드는 각 시나리오 뒤에 함께 둡니다 — *먼저 직접 시도한 다음* 비교해보세요.

> 💡 새 데이터지만 사용하는 도구는 똑같습니다. *오늘 배운 함수와 패턴을 그대로 적용*하면 됩니다.

## 시나리오 1 — 품질 진단 (Part 2 적용)

`quality_report_full()` 함수를 `bakery`에 그대로 적용해 다음을 답하세요.

1. 결측률이 가장 높은 컬럼은?
2. `maybe_datetime`이 True로 잡힌 컬럼이 있나요?
3. 수치형 중 IQR 이상치 비율이 가장 높은 컬럼은?

> **읽는 법:** `date_str`이 `maybe_datetime=True`로 잡혔는지 확인하세요. `quantity`에는 *50*이라는 큰 값이 있어 IQR 이상치 비율이 높게 나옵니다. `revenue`에 결측이 있어 결측률이 가장 높습니다.

## 시나리오 2 — 정제 파이프라인 (Part 3·6 적용)

다음 단계의 함수를 작성하고 `.pipe()`로 묶어 `bakery_clean`을 만드세요.

1. 완전 중복 제거
2. `store_id`의 앞뒤 공백 제거
3. `item`의 `'케익'` → `'케이크'`로 통일
4. `date_str`을 `date` 컬럼(datetime)으로 파싱
5. `revenue < 0`인 행은 `refunds_bakery`로 분리
6. `revenue` 결측 행 제거

> **읽는 법:** `.pipe()`를 6번 연달아 호출했더니 한 줄짜리 흐름으로 정제가 끝났습니다. **모두마켓에서 만든 패턴이 빵셀러에 그대로 적용**됩니다. 도구가 데이터에 의존하지 않는다는 뜻입니다.

## 시나리오 3 — 변환 + Parquet 저장 (Part 4·5 적용)

`bakery_clean`을 다음 두 KPI 표로 변환한 뒤, 각각 Parquet으로 저장하세요.

1. **월별·매장별 매출 합계** (행=매장, 열=월)
2. **상품별 평균 단가·총매출**

> **읽는 법:** 같은 도구로 *완전히 다른 데이터*에서 KPI 표가 나왔습니다. 이것이 *재사용 가능한 파이프라인*의 효용입니다.

## 종합 프로젝트 보고서 양식 (마크다운 셀로 채우세요)

아래 양식을 복사해 노트북 마지막에 마크다운 셀로 채우세요.

```markdown
# 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그
- 기간: 2025-04-01 ~ 2025-06-30 (3개월)
- 원본 행 수 / 정제 후 행 수: (   )행 → (   )행

## 2. 발견한 품질 문제
- 결측: (어느 컬럼 몇 %)
- 중복: (몇 건)
- 이상치: (어떤 값)
- 표기 혼재: (어느 컬럼 어떤 차이)

## 3. 처리 결정과 근거 (5줄로)
1. 중복 행 (   )건 → ...
2. store_id 공백 → ...
3. item '케익' → '케이크' → ...
4. revenue 음수 → 별도 보관 (   )건
5. revenue 결측 (   )건 → 제거 (이유: ...)

## 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월:
- (상품 관점) 매출 1위 상품:

## 5. 한계와 후속 작업
- (예) 환불 데이터의 분기별 발생 원인 분석 필요
- (예) item '케익'/'케이크' 외에 다른 표기 변종이 있는지 추가 점검 필요
```

# ✅ 오늘의 퀴즈

배운 내용을 잠깐 확인해볼게요. 틀려도 괜찮습니다. "이런 걸 배웠지" 떠올리는 용도예요.

### 개념 퀴즈

1. CSV와 Parquet 중 **자료형(dtype)** 을 저장 후 읽기에서도 보존하는 것은?
2. 다음 정제 단계의 순서로 가장 합리적인 것은? `(a) 결측 → 중복 → 이상치` `(b) 중복 → 문자열 → 날짜 → 이상치 → 결측` `(c) 이상치 → 결측 → 중복`
3. `.pipe(f)`와 `f(df)`는 어떤 점에서 같고, 어떤 점에서 다른가요? (가독성 측면)
4. 음수 금액(`amount = -50000`)을 **버리지 않고** 별도 변수로 분리한 이유는?

### 코드 퀴즈

`bakery_clean`에서 **`item`이 `'케이크'`이고 `revenue >= 10000`인 행만** 골라서 `store_id`별 평균 `revenue`를 구한 뒤, Parquet으로 저장하세요.

> **읽는 법:** `.query("...")` 는 문자열로 조건을 적는 또 다른 필터링 방법입니다(D+003에서 본 불리언 인덱싱과 동일한 결과). 마지막 `.to_parquet()` 한 줄로 *오늘 배운 흐름의 결승선* 까지 한 번에 갔습니다.

# 🎓 정리 & 다음 시간 예고

## 오늘 배운 것이 어떻게 이어졌나

```text
[Part 1] 인제스트(CSV)            → 자료형 손실이라는 출발점의 문제
   ↓  (어떻게 진단할까?)
[Part 2] 품질 리포트 함수         → D+002·003·005의 진단을 한 함수로
   ↓  (무엇을 어떤 순서로 고칠까?)
[Part 3] 정제 5단계               → D+003·005를 한 줄기로
   ↓  (분석 가능한 형태로 어떻게?)
[Part 4] 변환(병합·집계·피벗)     → D+004·006 적용
   ↓  (다음 분기에 어떻게 넘길까?)
[Part 5] Parquet 저장              → CSV의 한계를 해결
   ↓  (어떻게 재사용 가능하게?)
[Part 6] .pipe() 파이프라인        → D+006의 완성형
   ↓  (동료가 읽으려면?)
[Part 7] 판단 근거 문서화          → "왜"를 코드 옆에 남기는 습관
   ↓
[종합 프로젝트] 새 데이터(빵셀러)에 같은 흐름 적용
```

## 한 장 정리표

| 주제 | 핵심 한 줄 | 대표 코드 |
| --- | --- | --- |
| 인제스트 | 자료형 손실은 받는 즉시 발생 | `pd.read_csv` |
| 품질 리포트 | 매번 새 데이터, 같은 진단 양식 | `quality_report_full(df)` |
| 정제 순서 | 중복 → 문자열 → 날짜 → 이상치 → 결측 | 5개의 단계 함수 |
| 변환 | merge → groupby → pivot의 흐름 | `.merge().groupby().agg()` |
| 저장 | 자료형·압축·컬럼 읽기는 Parquet | `df.to_parquet(path)` |
| 파이프라인 | 흩어진 단계를 한 함수로 | `df.pipe(f).pipe(g)` |
| 문서화 | 결정의 *근거*를 표로 남긴다 | 결정 로그 DataFrame |

## 8일간 배운 도구 → 종합 프로젝트 어디서 썼나

| 출처 D+N | 도구 | 종합 프로젝트에서의 자리 |
| --- | --- | --- |
| D+002 | `info`·`describe` | 품질 리포트 함수의 기반 |
| D+003 | IQR · 결측 처리 | 이상치·결측 처리 단계 |
| D+004 | `merge` · `groupby` · pivot | 변환 Part |
| D+005 | `.str` · `pd.to_datetime` | 문자열·날짜 정제 단계 |
| D+006 | `.pipe()` · 함수 분리 | 파이프라인 함수화 |
| D+007 | Polars(선택) · dtype 최적화 | 대용량 시 같은 패턴 |
| D+008 | 검증·전달용 시각화 | 결과 시각화(후속 분석) |

## 🎓 다음 시간 예고

> **"지금까지 정제한 데이터, 어디서 왔는가?"**
>
> 오늘까지 우리는 *주어진 데이터*를 다듬는 법을 익혔습니다. 그러나 현실에서는 **데이터 자체를 구하는 일**이 분석의 출발점인 경우가 많습니다. 다음 모듈에서는 **웹 스크래핑**을 시작합니다 — HTML에서 표를 긁어오고, API로 호출해서 받아오는 *데이터의 윗단*을 배웁니다. 오늘 만든 파이프라인의 *입력*이 다음 모듈의 *출력*이 됩니다.

# 📝 오늘의 과제

오늘 만든 **end-to-end 파이프라인**을 정리해 GitHub 포트폴리오에 제출합니다. 이번 과제는 *모듈 종합 프로젝트*이라 8일간의 학습이 한 폴더로 마무리됩니다.

## 제출물

1. 빵셀러 종합 프로젝트을 끝까지 수행한 노트북(`.ipynb`)
2. 노트북 마지막에 **종합 프로젝트 보고서**(5개 항목) 마크다운 셀
3. 정제·변환 결과의 **Parquet 파일** 2개 (월·매장 / 상품별 KPI)
4. 결정 로그 표(마크다운 또는 DataFrame `to_markdown()` 출력)

## 필수 과제

- [ ] `quality_report_full()` 함수를 빵셀러 데이터에 적용한 결과를 노트북에 포함했다.
- [ ] 정제 5단계를 *작은 함수*로 나누고 `.pipe()`로 묶었다.
- [ ] `bakery_clean`을 Parquet으로 저장하고, 같은 데이터를 CSV로도 저장해 *용량 차이*를 한 줄로 적었다.
- [ ] 결정 로그 표(5단계 이상)를 채워 노트북에 박았다.
- [ ] 종합 프로젝트 보고서 5개 항목을 모두 채웠다.

## 심화 과제 (선택)

- [ ] `full_pipeline()`을 *모듈*(`.py` 파일)로 분리하고, 노트북에서는 `from pipeline import full_pipeline`으로 호출했다.
- [ ] 결정 로그 함수(`log_decision`)를 각 단계 함수 내부에서 자동 호출하도록 리팩터링했다.
- [ ] (선택) 같은 파이프라인을 **Polars** 로도 작성해 두 도구의 속도를 비교했다.

## 제출 방법 (GitHub)

이번 주는 PR 흐름을 권장합니다. 학습자 개인 공개 repo에 다음 구조로 올립니다.

```text
{학습자}/ai-data-bootcamp/
└── D009/
    ├── D009_capstone.ipynb
    ├── pipeline.py          # 심화 선택
    ├── outputs/
    │   ├── bakery_store_month.parquet
    │   └── bakery_item_kpi.parquet
    └── README.md            # 보고서를 README로 옮겨도 좋습니다
```

```bash
# 작업 브랜치에서 PR로 main에 제출
git checkout -b feature/d009-capstone
git add D009/
git commit -m "D009 종합 프로젝트: end-to-end 정제·변환·저장 파이프라인"
git push -u origin feature/d009-capstone
# GitHub UI에서 main으로 PR 생성, 강사가 줄 단위 리뷰
```

## 평가 기준

| 축 | 무엇을 보나 |
| --- | --- |
| 정확성 | 파이프라인이 처음부터 끝까지 오류 없이 실행되는가 |
| 합리성 | 각 결정의 *근거*가 코드 옆에 글로 남아 있는가 |
| 재사용성 | 함수가 *데이터에 의존하지 않게* 작성되었는가 |
| 산출물 품질 | Parquet·README·결정 로그가 GitHub에서 잘 보이는가 |

> 💡 모두의연구소의 과제는 순위를 매기지 않습니다. **"어제의 나보다 데이터를 더 신뢰할 수 있게 정제했는가"** 가 기준이에요. 이번 종합 프로젝트은 8일간의 성장을 한눈에 보여주는 산출물이 될 거예요.

---

수고하셨습니다! 🎉

오늘 여러분은 8일간 흩어져 있던 도구들을 **하나의 파이프라인**으로 묶었습니다. 더 중요한 건, *왜 그렇게 묶었는지*를 글로 남기는 습관을 들였다는 점입니다. 같은 코드를 6개월 뒤에 봐도, 동료가 처음 봐도, *결정의 흐름*이 그대로 살아 있는 노트북 — 그것이 데이터 분석가의 *진짜 산출물*입니다.

다음 모듈에서는 *그 파이프라인의 윗단*, 즉 데이터를 직접 구해 오는 **웹 스크래핑**에서 만나요. 천천히, 그러나 멈추지 않고 가봅시다. 🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 교안은 생성형 AI를 활용해 제작하고 제작자가 검수했습니다.<br>
무단 복제 및 배포를 금합니다.</sub>

In [ ]:
# 예제: 결정 로그를 리스트로 누적하다가 마지막에 표로 출력
decisions = []


def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})


# Part 3에서 했던 결정을 손으로 기록 (실무에서는 단계 함수 안에서 호출)
log_decision(
    "1. 중복 제거",
    "완전 중복 행 drop",
    "동일 키 동일 값은 시스템 입력 오류로 간주",
    "3건 제거",
)
log_decision(
    "2. 문자열 정제",
    "channel을 lower/strip",
    "'app '·'APP'은 'app'과 같은 채널",
    "채널 종류 4→2로 통일",
)
log_decision(
    "3. 날짜 파싱",
    "order_date를 datetime으로",
    "월별 집계가 분석 목적이므로 datetime 필요",
    f"NaT(파싱 실패): {orders_step3['order_date'].isna().sum()}건",
)
log_decision(
    "4. 이상치 처리",
    "음수 amount 분리, 수량 상위 1% 컷오프",
    "매출 분석이 목적이므로 환불은 별도 보관",
    f"환불 {len(refunds)}건, 수량 상위 컷오프 {q99}",
)
log_decision(
    "5. 결측치 처리",
    "amount 결측 행 제거",
    "결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",
    "최종 80건 내외 제거",
)

decision_log = pd.DataFrame(decisions)
decision_log

In [ ]:
# 예제: 결정 로그를 마크다운 표로 출력해 노트북 끝(또는 별도 파일)에 박기
md_table = "| 단계 | 액션 | 근거 | 결과 |\n|---|---|---|---|\n"
for d in decisions:
    md_table += f"| {d['step']} | {d['action']} | {d['reason']} | {d['result']} |\n"

print(md_table)

In [ ]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

In [ ]:
# 시나리오 1 — 모범 답안
qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

In [ ]:
# 시나리오 2 — 모범 답안
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True)
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip())
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"}))
def b_parse_date(df): return df.assign(
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"])

refunds_bakery = bakery[bakery["revenue"] < 0].copy()

bakery_clean = (
    bakery
    .pipe(b_dedup)
    .pipe(b_strip_store)
    .pipe(b_unify_item)
    .pipe(b_parse_date)
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외
    .pipe(lambda d: d.dropna(subset=["revenue"]))
    .reset_index(drop=True)
)

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")

In [ ]:
# 시나리오 3 — 모범 답안

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

In [ ]:
# 코드 퀴즈 — 모범 답안
cake_kpi = (
    bakery_clean
    .query("item == '케이크' and revenue >= 10000")
    .groupby("store_id")["revenue"]
    .mean()
    .round(0)
    .reset_index(name="avg_cake_revenue")
)
print(cake_kpi)

cake_kpi.to_parquet(OUT_DIR / "cake_kpi.parquet", index=False)
print(f"\n저장 완료: {OUT_DIR / 'cake_kpi.parquet'}")